# GDELT API Explorer

Interactive walkthrough of the GDELT 2.0 public REST APIs. Run each cell to see live responses.

**APIs covered:**
- **DOC API** — full-text article search (3-month window)
- **Context API** — sentence-level search (72-hour window)
- **GEO API** — geographic visualization (7-day window)
- **TV API** — television broadcast search (2009–present)

**Rate limit:** 1 request per 5 seconds. Each cell includes a delay to stay under the limit.

See `gdelt-api-reference.md` for the full written reference.

---
## Setup

In [ ]:
import requests
import time
import json
from datetime import datetime, timedelta
from IPython.display import display, HTML, Markdown

DOC_API = "https://api.gdeltproject.org/api/v2/doc/doc"
CONTEXT_API = "https://api.gdeltproject.org/api/v2/context/context"
GEO_API = "https://api.gdeltproject.org/api/v2/geo/geo"
TV_API = "https://api.gdeltproject.org/api/v2/tv/tv"

_last_request_time = 0

def gdelt_query(endpoint, params, label=""):
    """Make a GDELT API call with rate-limit handling."""
    global _last_request_time
    elapsed = time.time() - _last_request_time
    if elapsed < 10:
        time.sleep(10 - elapsed)

    for attempt in range(4):
        resp = requests.get(endpoint, params=params)
        _last_request_time = time.time()
        if resp.status_code != 429:
            break
        wait = 10 * (attempt + 1)
        print(f"Rate limited (attempt {attempt + 1}) — waiting {wait}s...")
        time.sleep(wait)

    resp.raise_for_status()

    if label:
        print(f"=== {label} ===")
    print(f"URL: {resp.url[:120]}..." if len(resp.url) > 120 else f"URL: {resp.url}")
    print(f"Status: {resp.status_code} | Size: {len(resp.content):,} bytes")

    try:
        return resp.json()
    except Exception:
        print(f"Response (not JSON): {resp.text[:200]}")
        return None


def show_articles(data, max_show=5):
    """Pretty-print artlist results."""
    articles = data.get("articles", []) if data else []
    print(f"\nTotal articles returned: {len(articles)}\n")
    for i, a in enumerate(articles[:max_show], 1):
        seen = a.get('seendate', '')
        try:
            dt = datetime.strptime(seen, '%Y%m%dT%H%M%SZ').strftime('%Y-%m-%d %H:%M')
        except (ValueError, TypeError):
            dt = seen
        print(f"  [{i}] {a.get('title', 'No title')}")
        print(f"      {a.get('domain', '?')} | {a.get('sourcecountry', '?')} | {dt}")
        print(f"      {a.get('url', '')}")
        print()
    if len(articles) > max_show:
        print(f"  ... and {len(articles) - max_show} more")


def show_json_sample(data, lines=20):
    """Print a compact JSON preview."""
    text = json.dumps(data, indent=2)
    sample = "\n".join(text.split("\n")[:lines])
    print(sample)
    total_lines = len(text.split("\n"))
    if total_lines > lines:
        print(f"  ... ({total_lines - lines} more lines)")


print("Setup complete.")

---
## 1. DOC API — Basic Article Search (`artlist`)

The most common use case: search for articles by keyword, get back a list with metadata.

**Key facts:**
- Spaces between words = implicit AND (all words must appear in article)
- No stemming: `regulate` won't match `regulation`
- Max 250 results per query
- Rolling 3-month window

In [2]:
data = gdelt_query(DOC_API, {
    "query": "medical device sourcelang:english",
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "7d",
    "format": "json",
    "sort": "hybridrel",
}, label="Basic keyword search")

show_articles(data)

Rate limited (attempt 1) — waiting 10s...
=== Basic keyword search ===
URL: https://api.gdeltproject.org/api/v2/doc/doc?query=medical+device+sourcelang%3Aenglish&mode=artlist&maxrecords=5&timespan...
Status: 200 | Size: 2,565 bytes

Total articles returned: 5

  [1] WONBIOGEN Co ., Ltd : Won BioGen Accelerates Global Market Expansion with Advanced Moist Wound Care Technology
      finanznachrichten.de | Germany | 2026-02-24 10:15
      https://www.finanznachrichten.de/nachrichten-2026-02/67773243-wonbiogen-co-ltd-won-biogen-accelerates-global-market-expansion-with-advanced-moist-wound-care-technology-008.htm

  [2] Ray J Says Fan Swiped His Heart Monitor , Begs For Its Return
      allhiphop.com | United States | 2026-02-20 23:30
      https://allhiphop.com/news/ray-j-says-fan-swiped-his-heart-monitor-begs-for-its-return/

  [3] Herz P1 Smart Band 2026 : What It Tracks , How It Works , and What Consumers Should Know Before Buying
      portal.sina.com.hk | Hong Kong | 2026-02-27 08:45


### Inspect the raw JSON response

Each article has: `url`, `url_mobile`, `title`, `seendate`, `socialimage`, `domain`, `language`, `sourcecountry`.

That's it — no body text, no sentiment score, no themes. The DOC API is for **discovery**, not deep metadata.

In [3]:
if data and data.get("articles"):
    print("Fields in each article:")
    print(json.dumps(data["articles"][0], indent=2))
else:
    print("No data — try re-running the cell above first.")

Fields in each article:
{
  "url": "https://www.finanznachrichten.de/nachrichten-2026-02/67773243-wonbiogen-co-ltd-won-biogen-accelerates-global-market-expansion-with-advanced-moist-wound-care-technology-008.htm",
  "url_mobile": "",
  "title": "WONBIOGEN Co ., Ltd : Won BioGen Accelerates Global Market Expansion with Advanced Moist Wound Care Technology",
  "seendate": "20260224T101500Z",
  "socialimage": "https://www.finanznachrichten.de/chart-biogen-inc-aktie-intraklein-tradegate.png",
  "domain": "finanznachrichten.de",
  "language": "English",
  "sourcecountry": "Germany"
}


---
## 2. Query Operators

All operators go inside the `query` parameter as a single string. Let's try each one.

### 2a. Exact Phrase (`"quoted"`)

Words in quotes must appear **adjacent and in order**. Compare with the implicit AND above.

In [4]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" sourcelang:english',
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "7d",
    "format": "json",
    "sort": "hybridrel",
}, label='Exact phrase: "medical device"')

show_articles(data)

=== Exact phrase: "medical device" ===
URL: https://api.gdeltproject.org/api/v2/doc/doc?query=%22medical+device%22+sourcelang%3Aenglish&mode=artlist&maxrecords=5&ti...
Status: 200 | Size: 2,463 bytes

Total articles returned: 5

  [1] WONBIOGEN Co ., Ltd : Won BioGen Accelerates Global Market Expansion with Advanced Moist Wound Care Technology
      finanznachrichten.de | Germany | 2026-02-24 10:15
      https://www.finanznachrichten.de/nachrichten-2026-02/67773243-wonbiogen-co-ltd-won-biogen-accelerates-global-market-expansion-with-advanced-moist-wound-care-technology-008.htm

  [2] Yogi in Japan : What on UP CM investment pitch as he meets top firms in Tokyo | Lucknow News
      indianexpress.com | India | 2026-02-25 07:30
      https://indianexpress.com/article/cities/lucknow/yogi-in-japan-whats-on-up-cms-investment-pitch-as-he-meets-top-firms-in-tokyo-10551111/

  [3] TTRX Stock Price Quote | Morningstar
      morningstar.com | China | 2026-02-26 02:00
      https://www.morningstar

### 2b. Boolean OR

Use `(a OR b OR c)` — must be capitalized, wrapped in parens. At least one must match.

Combine with AND by just adding more terms outside the parens.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '(medtronic OR "boston scientific" OR stryker) recall sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label="Boolean OR — manufacturers + recall")

show_articles(data)

### 2c. Exclusion (`-term`)

Prefix with minus to exclude articles containing that term.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" -politics -election sourcelang:english',
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "7d",
    "format": "json",
    "sort": "hybridrel",
}, label="Exclusion: -politics -election")

show_articles(data)

### 2d. Proximity Search (`near#:`)

The two words must appear within N words of each other. Much more precise than simple AND.

Syntax: `near10:"word1 word2"` — word1 and word2 within 10 words.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": 'near10:"FDA recall" "medical device" sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label="Proximity: FDA within 10 words of recall")

show_articles(data)

### 2e. Repeat Threshold (`repeat#:`)

Article must mention the term at least N times. Articles that repeat your keyword are more likely to be *about* it rather than just mentioning it in passing.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": 'repeat3:"medical device" sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label='Repeat: "medical device" at least 3 times')

show_articles(data)

---
## 3. Metadata Filters

These go inside the `query` string alongside your keywords.

### 3a. Source Country (`sourcecountry:`)

Filter by where the news outlet is physically based. Name with spaces removed and lowercase, or FIPS 2-char code.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" recall sourcecountry:unitedstates sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label="Source country: United States only")

show_articles(data)

### 3b. Domain Filter (`domain:` vs `domainis:`)

- `domain:cnn.com` — partial match (also matches subdomain.cnn.com)
- `domainis:cnn.com` — exact match only

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" domain:reuters.com sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "90d",
    "format": "json",
    "sort": "datedesc",
}, label="Domain filter: reuters.com")

show_articles(data)

### 3c. Tone / Sentiment Filter (`tone<N`, `tone>N`)

Filter by article sentiment score. Negative = critical coverage, positive = favorable.

Note: this filters *before* returning results — you only get articles matching the tone threshold.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" tone<-5 sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "30d",
    "format": "json",
    "sort": "toneasc",
}, label="Tone filter: strongly negative (tone < -5)")

show_articles(data)

### 3d. Theme Filter (`theme:`)

GDELT tags articles with GKG themes. You can filter by theme to find articles in specific categories.

Full theme list: `http://data.gdeltproject.org/api/v2/guides/LOOKUP-GKGTHEMES.TXT`

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" theme:HEALTH_PANDEMIC sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "90d",
    "format": "json",
    "sort": "datedesc",
}, label="Theme filter: HEALTH_PANDEMIC")

show_articles(data)

---
## 4. Timeline Modes

Instead of returning articles, timeline modes return **time-series data** showing how coverage changes over time.

Resolution adapts automatically:
- < 72 hours → 15-minute intervals
- 72h – 1 week → hourly intervals  
- \> 1 week → daily intervals

### 4a. `timelinevol` — Coverage as % of Total GDELT

Value is a percentage of all articles GDELT monitored at that timestep. Useful for relative trends but not absolute counts.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" sourcelang:english',
    "mode": "timelinevol",
    "timespan": "30d",
    "format": "json",
}, label="timelinevol — % of total GDELT coverage")

if data:
    print(f"\nSeries: {data['timeline'][0]['series']}")
    print(f"Resolution: {data['query_details']['date_resolution']}")
    points = data['timeline'][0]['data']
    print(f"Data points: {len(points)}")
    print(f"\nFirst 5:")
    for p in points[:5]:
        print(f"  {p['date']}  value={p['value']}")

### 4b. `timelinevolraw` — Raw Article Counts

Same as timelinevol but returns actual article counts plus the total monitored (`norm` field).

This is the most useful timeline mode for understanding absolute volume.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" sourcelang:english',
    "mode": "timelinevolraw",
    "timespan": "30d",
    "format": "json",
}, label="timelinevolraw — raw article counts")

if data:
    points = data['timeline'][0]['data']
    print(f"\nSeries: {data['timeline'][0]['series']}")
    print(f"Data points: {len(points)}")
    print(f"\nSample ('norm' = total articles GDELT monitored that timestep):")
    for p in points[:5]:
        print(f"  {p['date']}  articles={p['value']}  total_monitored={p.get('norm', '?')}")

    values = [p['value'] for p in points]
    total = sum(values)
    peak = max(values)
    peak_date = points[values.index(peak)]['date']
    print(f"\nTotal: {total} | Peak: {peak} at {peak_date} | Avg: {total / len(values):.1f}/timestep")

### 4c. Visualize the Timeline (matplotlib)

Let's plot the raw article counts to see the coverage pattern.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not installed — run: pip install matplotlib")
    print("Skipping plot.")

if HAS_MPL and data:
    points = data['timeline'][0]['data']
    dates = [datetime.strptime(p['date'], '%Y%m%dT%H%M%SZ') for p in points]
    values = [p['value'] for p in points]
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(dates, values, width=0.8 if len(dates) < 60 else 0.03, color='steelblue', alpha=0.8)
    ax.set_ylabel('Article Count')
    ax.set_title('"medical device" — Article Volume (past 30 days)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

### 4d. `timelinetone` — Sentiment Over Time

Average emotional tone of matching articles at each timestep. Negative = critical, positive = favorable.

In [ ]:
data_tone = gdelt_query(DOC_API, {
    "query": '"medical device" sourcelang:english',
    "mode": "timelinetone",
    "timespan": "30d",
    "format": "json",
}, label="timelinetone — sentiment over time")

if data_tone:
    points = data_tone['timeline'][0]['data']
    tones = [p['value'] for p in points if p['value'] != 0]
    if tones:
        print(f"\nNon-zero tone values: {len(tones)}")
        print(f"Avg tone: {sum(tones)/len(tones):.2f}")
        print(f"Min tone: {min(tones):.2f} (most negative)")
        print(f"Max tone: {max(tones):.2f} (most positive)")
    else:
        print("All zero values — query may be too narrow for tone data.")

In [ ]:
if HAS_MPL and data_tone:
    points = data_tone['timeline'][0]['data']
    dates = [datetime.strptime(p['date'], '%Y%m%dT%H%M%SZ') for p in points]
    tones = [p['value'] for p in points]

    fig, ax = plt.subplots(figsize=(14, 4))
    colors = ['firebrick' if t < 0 else 'forestgreen' for t in tones]
    ax.bar(dates, tones, width=0.03, color=colors, alpha=0.7)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.set_ylabel('Average Tone')
    ax.set_title('"medical device" — Sentiment Over Time (past 30 days)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

### 4e. `timelinesourcecountry` — Coverage by Country

Breaks the timeline into **one series per source country**. Shows where the coverage is coming from.

In [ ]:
data_country = gdelt_query(DOC_API, {
    "query": '"medical device" sourcelang:english',
    "mode": "timelinesourcecountry",
    "timespan": "30d",
    "format": "json",
}, label="timelinesourcecountry — coverage by source country")

if data_country:
    print(f"\nNumber of country series: {len(data_country['timeline'])}")
    print("\nCountries reporting on 'medical device':")
    
    # Rank countries by total volume
    country_totals = []
    for series in data_country['timeline']:
        total = sum(p['value'] for p in series['data'])
        country_totals.append((series['series'], total))
    
    country_totals.sort(key=lambda x: x[1], reverse=True)
    for name, total in country_totals[:15]:
        bar = '█' * int(total * 100)  # simple text bar chart
        print(f"  {name:30s} {total:.4f}  {bar}")

### 4f. `timelinelang` — Coverage by Language

Same idea but broken down by the **original language** of the article (before GDELT's machine translation).

In [ ]:
data_lang = gdelt_query(DOC_API, {
    "query": '"medical device"',  # no sourcelang filter — see all languages
    "mode": "timelinelang",
    "timespan": "30d",
    "format": "json",
}, label="timelinelang — coverage by language")

if data_lang:
    print(f"\nNumber of language series: {len(data_lang['timeline'])}")
    print("\nLanguages covering 'medical device':")
    
    lang_totals = []
    for series in data_lang['timeline']:
        total = sum(p['value'] for p in series['data'])
        lang_totals.append((series['series'], total))
    
    lang_totals.sort(key=lambda x: x[1], reverse=True)
    for name, total in lang_totals[:10]:
        bar = '█' * int(total * 100)
        print(f"  {name:25s} {total:.4f}  {bar}")

---
## 5. Sorting Options

The `sort` parameter controls result ordering. Available values:

- `hybridrel` — relevance (default, best match first)
- `datedesc` — newest first
- `toneasc` — most negative tone first
- `tonedesc` — most positive tone first

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" recall sourcelang:english',
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label="Sort: datedesc (newest first)")

show_articles(data)

---
## 6. Time Parameters

Two ways to specify time:
- **Relative**: `timespan=7d` (past 7 days from now)
- **Absolute**: `startdatetime=20260201000000&enddatetime=20260215235959`

Both must be within the rolling 3-month window.

In [ ]:
now = datetime.utcnow()
start = now.replace(day=1, hour=0, minute=0, second=0)
end = start + timedelta(days=14)

start_str = start.strftime('%Y%m%d%H%M%S')
end_str = end.strftime('%Y%m%d%H%M%S')
print(f"Searching: {start_str} to {end_str}")

data = gdelt_query(DOC_API, {
    "query": '"medical device" recall sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "startdatetime": start_str,
    "enddatetime": end_str,
    "format": "json",
    "sort": "datedesc",
}, label="Absolute date range: first 2 weeks of this month")

show_articles(data)

---
## 7. Context API — Sentence-Level Search

The Context API searches **individual sentences**, not whole articles. All search terms must appear
in the **same sentence**. Returns the matching sentence as a snippet.

**Limitations:** 72-hour window only, max 200 results.

In [ ]:
data_ctx = gdelt_query(CONTEXT_API, {
    "query": '"medical device" recall',
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "72h",
    "format": "json",
    "sort": "datedesc",
}, label="Context API — sentence-level search")

if data_ctx:
    show_json_sample(data_ctx, lines=30)

In [ ]:
if data_ctx and data_ctx.get("articles"):
    for i, a in enumerate(data_ctx["articles"][:5], 1):
        print(f"[{i}] {a.get('title', 'No title')}")
        print(f"    Domain: {a.get('domain', '?')}")
        for key in ['context', 'contexthtml', 'excerpt']:
            if key in a:
                text = a[key][:200] + '...' if len(a.get(key, '')) > 200 else a.get(key, '')
                print(f"    Snippet: {text}")
                break
        print(f"    URL: {a.get('url', '')}")
        print()

### Context API: `isquote=1` — Only Direct Quotes

Filter to only return sentences that are direct quotes from people.

In [ ]:
data_quotes = gdelt_query(CONTEXT_API, {
    "query": '"medical device"',
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "72h",
    "format": "json",
    "isquote": 1,
}, label="Context API — direct quotes only")

if data_quotes and data_quotes.get("articles"):
    for i, a in enumerate(data_quotes["articles"][:5], 1):
        print(f"[{i}] {a.get('title', 'No title')}")
        print(f"    {a.get('domain', '?')} | {a.get('url', '')}")
        print()
else:
    print("No quoted sentences found — try a broader query or longer timespan.")

---
## 8. GEO API — Geographic Data

Returns geographic locations mentioned in articles matching your query. Rolling **7-day window only**.

Supports GeoJSON output for mapping.

In [ ]:
data_geo = gdelt_query(GEO_API, {
    "query": '"medical device" recall sourcelang:english',
    "mode": "pointdata",
    "format": "geojson",
    "timespan": "7d",
    "geores": 2,
}, label="GEO API — locations in medical device recall coverage")

if data_geo and data_geo.get("features"):
    features = data_geo["features"]
    print(f"\nLocations found: {len(features)}\n")
    for f in features[:10]:
        props = f.get("properties", {})
        coords = f.get("geometry", {}).get("coordinates", [None, None])
        name = props.get("name", "Unknown")
        print(f"  {name} ({coords[1]:.2f}, {coords[0]:.2f})" if coords[0] else f"  {name}")
else:
    print("No geographic data returned.")
    if data_geo:
        show_json_sample(data_geo, lines=15)

---
## 9. TV API — Television Broadcast Monitoring

Searches closed captions from 163+ US TV stations. Archive goes back to **July 2009**.

Much longer time window than DOC API (up to 2 years).

In [ ]:
data_tv = gdelt_query(TV_API, {
    "query": '"medical device" market:"National"',
    "mode": "timelinevol",
    "format": "json",
    "timespan": "6m",
    "datanorm": "perc",
    "datacomb": "combined",
}, label="TV API — national broadcast mentions (6 months)")

if data_tv and data_tv.get("timeline"):
    for series in data_tv["timeline"]:
        points = series.get("data", [])
        print(f"\nSeries: {series.get('series', '?')} | Points: {len(points)}")
        nonzero = [p for p in points if p.get('value', 0) > 0]
        print(f"Non-zero data points: {len(nonzero)}")
        for p in nonzero[:5]:
            print(f"  {p['date']}  value={p['value']}")
else:
    print("No TV data returned.")
    if data_tv:
        show_json_sample(data_tv, lines=15)

In [ ]:
data_clips = gdelt_query(TV_API, {
    "query": '"medical device"',
    "mode": "clipgallery",
    "format": "json",
    "timespan": "1m",
    "maxrecords": 5,
}, label="TV API — clip gallery")

if data_clips and data_clips.get("clips"):
    for i, clip in enumerate(data_clips["clips"][:5], 1):
        print(f"[{i}] {clip.get('show', '?')} on {clip.get('station', '?')}")
        print(f"    Date: {clip.get('date', '?')}")
        print(f"    Snippet: {clip.get('snippet', 'N/A')[:150]}")
        print()
else:
    print("No clips found.")
    if data_clips:
        show_json_sample(data_clips, lines=20)

---
## 10. Medical Device Examples

Putting it all together — a few example queries relevant to our use case.

### 10a. Recall/Safety Events — Volume Over Time

Use `timelinevolraw` to see how many articles match a safety-focused query.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" (recall OR warning OR alert) sourcelang:english',
    "mode": "timelinevolraw",
    "timespan": "30d",
    "format": "json",
}, label="Medical device safety events — 30-day volume")

if data and data.get("timeline"):
    points = data['timeline'][0]['data']
    values = [p['value'] for p in points]
    total = sum(values)
    peak = max(values)
    peak_date = points[values.index(peak)]['date']
    print(f"\nTotal: {total:,} | Peak: {peak} at {peak_date} | Avg: {total / 30:.1f}/day")

### 10b. Manufacturer-Specific Search

Combine manufacturer names with OR, plus an event keyword.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '(medtronic OR "boston scientific" OR stryker) (recall OR warning OR "FDA") sourcelang:english',
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label="Manufacturers + recall/FDA")

show_articles(data)

### 10c. Domain Quality Check

What domains are appearing in our results? This helps us decide if we need domain filtering.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" (recall OR warning OR alert) sourcelang:english',
    "mode": "artlist",
    "maxrecords": 250,
    "timespan": "30d",
    "format": "json",
    "sort": "datedesc",
}, label="Domain analysis — 250 articles")

if data and data.get("articles"):
    articles = data["articles"]
    from collections import Counter
    domains = Counter(a.get('domain', '?') for a in articles)
    countries = Counter(a.get('sourcecountry', '?') for a in articles)

    print(f"\n--- Top 20 Domains ---")
    for domain, count in domains.most_common(20):
        bar = '█' * count
        print(f"  {domain:35s} {count:3d}  {bar}")

    print(f"\n--- Source Countries ---")
    for country, count in countries.most_common(10):
        bar = '█' * count
        print(f"  {country:25s} {count:3d}  {bar}")

### 10d. No-Stemming Workaround

GDELT has no stemming — `recall` won't match `recalled` or `recalls`. Use OR to cover variants.

In [ ]:
data = gdelt_query(DOC_API, {
    "query": '"medical device" (recall OR recalled OR recalls) sourcelang:english',
    "mode": "timelinevolraw",
    "timespan": "30d",
    "format": "json",
}, label="Stemming workaround — recall/recalled/recalls")

if data and data.get("timeline"):
    points = data['timeline'][0]['data']
    total = sum(p['value'] for p in points)
    print(f"\nTotal articles (30d): {total:,}")

### 10e. Combining Strategies

Build a comprehensive monitoring query that covers multiple angles.

In [ ]:
comprehensive_query = (
    '("medical device" OR "medical devices") '
    '(recall OR recalled OR recalls OR warning OR alert OR "adverse event" OR defect OR malfunction OR shortage) '
    'sourcelang:english'
)

print(f"Query: {comprehensive_query}\n")

data = gdelt_query(DOC_API, {
    "query": comprehensive_query,
    "mode": "artlist",
    "maxrecords": 10,
    "timespan": "7d",
    "format": "json",
    "sort": "datedesc",
}, label="Comprehensive medical device events query")

show_articles(data, max_show=10)

In [ ]:
# Check volume for the comprehensive query over 3 months
data_vol = gdelt_query(DOC_API, {
    "query": comprehensive_query,
    "mode": "timelinevolraw",
    "timespan": "3m",
    "format": "json",
}, label="Comprehensive query — 3-month volume")

if data_vol and data_vol.get("timeline"):
    points = data_vol['timeline'][0]['data']
    values = [p['value'] for p in points]
    total = sum(values)
    peak = max(values)
    peak_date = points[values.index(peak)]['date']
    
    print(f"\nTotal articles (3 months): {total:,}")
    print(f"Average per day: {total / 90:.1f}")
    print(f"Peak: {peak} articles at {peak_date}")
    
    # Plot if matplotlib available
    if HAS_MPL:
        dates = [datetime.strptime(p['date'], '%Y%m%dT%H%M%SZ') for p in points]
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar(dates, values, width=0.8, color='steelblue', alpha=0.8)
        ax.set_ylabel('Article Count')
        ax.set_title('Medical Device Events — 3-Month Volume')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()

---
## 11. CSV Export

The DOC API also supports CSV format directly. Useful for quick exports without post-processing.

In [ ]:
elapsed = time.time() - _last_request_time
if elapsed < 10:
    time.sleep(10 - elapsed)

resp = requests.get(DOC_API, params={
    "query": '"medical device" recall sourcelang:english',
    "mode": "artlist",
    "maxrecords": 5,
    "timespan": "7d",
    "format": "csv",
    "sort": "datedesc",
})
_last_request_time = time.time()

print(f"Status: {resp.status_code}")
print(f"\nRaw CSV response (first 1000 chars):")
print(resp.text[:1000])

---
## 12. Handling the 250-Result Cap

The DOC API returns max 250 articles per request. To collect more, break the time range into
smaller windows and query each separately.

Here's the pattern:
1. Use `timelinevolraw` to see volume distribution
2. Break into windows where each window has < 250 articles
3. Query each window with `startdatetime`/`enddatetime`
4. Deduplicate by URL

In [ ]:
data_plan = gdelt_query(DOC_API, {
    "query": '"medical device" sourcelang:english',
    "mode": "timelinevolraw",
    "timespan": "14d",
    "format": "json",
}, label="Volume check for windowing strategy")

if data_plan and data_plan.get("timeline"):
    points = data_plan['timeline'][0]['data']

    from collections import defaultdict
    daily = defaultdict(int)
    for p in points:
        day = p['date'][:8]
        daily[day] += p['value']

    print(f"\n{'Date':<12} {'Articles':>10} {'Needs windowing?':>20}")
    print("-" * 44)
    for day, count in sorted(daily.items()):
        flag = " ← YES" if count > 250 else ""
        print(f"{day:<12} {count:>10}{flag}")

    total = sum(daily.values())
    print(f"\nTotal: {total} | Max daily: {max(daily.values())}")
    if max(daily.values()) <= 250:
        print("→ Daily windows would work — no day exceeds 250.")
    else:
        print("→ Some days exceed 250 — would need sub-day windows.")

---
## 13. Rate Limit Behavior

What actually happens when you exceed the rate limit?

In [ ]:
params = {
    "query": "test sourcelang:english",
    "mode": "artlist",
    "maxrecords": 1,
    "timespan": "1d",
    "format": "json",
}

resp1 = requests.get(DOC_API, params=params)
print(f"Request 1: status={resp1.status_code}")

resp2 = requests.get(DOC_API, params=params)
print(f"Request 2 (immediate): status={resp2.status_code}")

if resp2.status_code == 429:
    print(f"\nRate limited! Response: {resp2.text[:200]}")
else:
    print("\nNo rate limit hit (server was lenient this time)")

_last_request_time = time.time()

---
## Quick Reference: Building Queries

| Want to... | Query syntax |
|---|---|
| All words must appear | `medical device recall` (spaces = AND) |
| Exact phrase | `"medical device"` |
| At least one of several | `(recall OR warning OR alert)` |
| Exclude a term | `-politics` |
| Words near each other | `near10:"FDA recall"` |
| Term appears N+ times | `repeat3:"medical device"` |
| English articles only | `sourcelang:english` |
| From specific country | `sourcecountry:unitedstates` |
| From specific domain | `domainis:fda.gov` |
| Negative coverage | `tone<-5` |
| High-emotion coverage | `toneabs>10` |
| Tagged with GKG theme | `theme:HEALTH_PANDEMIC` |

**Combine freely:** `"medical device" (recall OR warning) sourcecountry:unitedstates tone<-3 sourcelang:english`